# Generate explicit SCD Type 2 history

Offline path: use the included metadata-only fixture. Databricks path: use the active `spark` session. No credentials or source records are embedded.

## Prerequisites

Install `great-generator`. For SDK access outside a notebook, install `great-generator[databricks]` and use Databricks unified authentication. Metadata visibility and source object permissions are controlled by Unity Catalog.

In [ ]:
from pathlib import Path

from great_generator import StaticUnityCatalogClient

SOURCE_TABLE = "retail.sales.fact_orders"
FIXTURE = Path("examples/databricks/unity_catalog/mock_uc_metadata.json")
client = StaticUnityCatalogClient.from_path(FIXTURE)

# In a Databricks workspace, remove client=client and pass spark=spark instead.


## Generate synthetic data from metadata

In [ ]:
from great_generator import generate_relational_from_unity_catalog

datasets = generate_relational_from_unity_catalog(
    "retail.sales.dim_customer",
    include_related=False,
    model_type="scd2",
    scd2_config={
        "natural_key": ["customer_id"],
        "surrogate_key": "customer_sk",
        "effective_start": "effective_start_date",
        "effective_end": "effective_end_date",
        "current_flag": "is_current",
    },
    rows=25,
    seed=42,
    client=client,
)
history = datasets["dim_customer_history"]
history.head()


## Validate

In [ ]:
assert history.groupby("customer_id")["is_current"].sum().eq(1).all()
assert (history["effective_start_date"] < history["effective_end_date"]).all()


## Cleanup and limitations

These examples do not write tables. Remove only sandbox artifacts you explicitly create. Generated data is synthetic, not anonymized production data. Complex types and cyclic/self-referencing relationships require explicit modeling.